# Lesson 2: BoTorch data and scaling for a self-driving lab

This lesson connects BoTorch's data conventions to the SDL experiment loop. We will prepare experimental observations, keep model coordinates consistent, and trace how a proposed recipe can be sent to a lab system and its measured result returned to the model.

1. Tensor shapes for experiment inputs and outcomes.
2. Normalize inputs and convert a proposal back to physical units.
3. Input transforms versus helper functions.
4. Outcome standardization and known measurement variance.
5. The BoTorch-to-SDL-to-BoTorch data path.

The examples use salt concentration, EC fraction, and DMC fraction. The SDL interface below is illustrative; adapt the payload fields to your actual instrument/workflow API.

## 1. The basic data shapes

For `n` experiments, `d` input variables, and one measured outcome:

- `train_X` has shape `n x d`: one experiment per row, one variable per column.
- `train_Y` has shape `n x 1`: one outcome for each experiment.

For `m` outcomes, `train_Y` is generally `n x m`. Rows in `train_X` and `train_Y` must refer to the same experiments in the same order.

In [57]:
import torch

torch.set_default_dtype(torch.double)

train_X = torch.tensor([
    [0.10, 0.20, 0.70],
    [0.20, 0.30, 0.50],
    [0.30, 0.20, 0.50],
    [0.40, 0.30, 0.30],
    [0.50, 0.20, 0.30],
])
train_Y = torch.tensor([
    [1.10],
    [1.35],
    [1.42],
    [1.60],
    [1.48],
])

print("train_X:", train_X.shape)  # n x d = 5 x 3
print("train_Y:", train_Y.shape)  # n x 1

train_X: torch.Size([5, 3])
train_Y: torch.Size([5, 1])


### Candidate shape: what does `q` mean?

For model inputs and acquisition functions, the common shape is `batch_shape x q x d`:

- `d`: number of input variables.
- `q`: number of candidate points considered jointly.
- `batch_shape`: optional leading dimensions for independent batches of inputs.

For example, a single proposed experiment with three variables has shape `1 x 3` (`q=1`, `d=3`). When scoring several separate one-point candidates, each is represented as a batch item with shape `batch x 1 x d`. Keeping the candidate dimension explicit prevents confusing “three candidates” with “one candidate having three variables.”

In [58]:
# One candidate, three input variables: q x d = 1 x 3.
candidate = torch.tensor([[0.20, 0.30, 0.50]])
print("One candidate:", candidate.shape)

# Three separate candidates, each with q=1 and d=3: batch x q x d.
candidates = torch.tensor([
    [[0.10, 0.20, 0.70]],
    [[0.30, 0.20, 0.50]],
    [[0.50, 0.20, 0.30]],
])
print("Batched one-point candidates:", candidates.shape)

One candidate: torch.Size([1, 3])
Batched one-point candidates: torch.Size([3, 1, 3])


## 2. Normalize input variables for the search space

GPs work best when input dimensions are on comparable scales. A common choice is mapping each allowed input range to `[0, 1]`:

`x_normalized = (x - lower) / (upper - lower)`

For multiple variables, provide bounds as a `2 x d` tensor: the first row contains lower bounds and the second row upper bounds. Use the same coordinate system consistently for training inputs, prediction inputs, and acquisition-optimizer bounds. For an SDL, keep the physical bounds and units too: the model's normalized candidate must be converted back before a recipe is prepared.

In [59]:
# Example physical bounds: salt concentration (M), EC fraction, DMC fraction.
bounds = torch.tensor([
    [0.5, 0.0, 0.0],
    [2.0, 1.0, 1.0],
])
X_physical = torch.tensor([
    [0.8, 0.3, 0.7],
    [1.4, 0.45, 0.55],
])

from botorch.utils.transforms import normalize, unnormalize

X_normalized = normalize(X_physical, bounds)
X_recovered = unnormalize(X_normalized, bounds)

print("Normalized inputs:\n", X_normalized)
print("Recovered physical inputs:\n", X_recovered)
print("Round trip correct:", torch.allclose(X_physical, X_recovered))

Normalized inputs:
 tensor([[0.2000, 0.3000, 0.7000],
        [0.6000, 0.4500, 0.5500]])
Recovered physical inputs:
 tensor([[0.8000, 0.3000, 0.7000],
        [1.4000, 0.4500, 0.5500]])
Round trip correct: True


### `normalize()` versus `Normalize`

These are related but different BoTorch APIs:

- `normalize(X, bounds)` is a helper function. Call it yourself on training data and on each new candidate; use normalized coordinates consistently afterward.
- `Normalize(d=..., bounds=...)` is a model input transform. Pass it to the model and provide model inputs in the original physical coordinates; the model applies the transform internally.

Choose one approach and normalize exactly once. With a model input transform, keep candidates and optimizer bounds in the same physical coordinates as the model inputs. With manual normalization, use normalized coordinates for all three.

In [60]:
from botorch.models.transforms import Normalize

input_transform = Normalize(d=3, bounds=bounds)
model_inputs_normalized_by_transform = input_transform(X_physical)
print("Transform output:\n", model_inputs_normalized_by_transform)

Transform output:
 tensor([[0.2000, 0.3000, 0.7000],
        [0.6000, 0.4500, 0.5500]])


## 3. Standardize the outcome

Input normalization rescales each **input variable** to a chosen range. Outcome standardization rescales the **observed objective** using its mean and standard deviation:

`Y_standardized = (Y - mean(Y)) / std(Y)`

In this BoTorch version, `SingleTaskGP` standardizes outcomes by default when no explicit outcome transform is provided, then returns posterior predictions in the original outcome scale. Keep recorded experimental `train_Y` values in their meaningful physical/scientific units; do not manually standardize them as well unless you deliberately disable or replace the model's transform.

In [61]:
from botorch.models.transforms import Standardize

outcome_transform = Standardize(m=1)  # m is the number of outcomes
print(outcome_transform)

# Explicit form (optional; this is the default for a single outcome in this setup):
# model = SingleTaskGP(train_X, train_Y, outcome_transform=Standardize(m=1))

Standardize()


## 4. Known measurement noise: `train_Yvar`

If measurement uncertainty is known or estimated from replicates, `train_Yvar` supplies the **variance** of each observation (not its standard deviation). For one outcome it has shape `n x 1`, matching `train_Y`; values must be non-negative and use the squared units of the outcome. For example, a measurement standard deviation of `0.05` corresponds to variance `0.05**2 = 0.0025`.

When `train_Yvar` is omitted, the model estimates a common observation-noise level. Supply it only when you have defensible per-observation noise estimates; it represents measurement noise, not the GP's uncertainty about the unknown function.

In [62]:
# Example only: variance estimates for the five observations above.
train_Yvar = torch.tensor([
    [0.0025],
    [0.0016],
    [0.0036],
    [0.0025],
    [0.0049],
])
print("train_Yvar:", train_Yvar.shape)

train_Yvar: torch.Size([5, 1])


## 5. A complete small example

The next cells use physical variables for salt concentration, EC fraction, and DMC fraction. We manually normalize the inputs, fit a `SingleTaskGP`, and predict at one new physical formulation. This is the model side of an SDL workflow; the following section makes the proposal/measurement handoff explicit.

In [63]:
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

# Columns: salt concentration, EC fraction, DMC fraction.
physical_bounds = torch.tensor([
    [0.5, 0.0, 0.0],
    [2.0, 1.0, 1.0],
])
train_X_physical = torch.tensor([
    [0.8, 0.3, 0.7],
    [1.0, 0.4, 0.6],
    [1.2, 0.5, 0.5],
    [1.5, 0.4, 0.6],
    [1.8, 0.6, 0.4],
])
train_Y = torch.tensor([[1.20], [1.35], [1.50], [1.58], [1.42]])

train_X = normalize(train_X_physical, physical_bounds)
model = SingleTaskGP(train_X, train_Y)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

print("Training input shape:", train_X.shape)
print("Training outcome shape:", train_Y.shape)

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([1.4100]), std = tensor([0.1456])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


Training input shape: torch.Size([5, 3])
Training outcome shape: torch.Size([5, 1])


In [64]:
# Candidate arrives in physical units, so normalize it before prediction.
candidate_physical = torch.tensor([[1.4, 0.45, 0.55]])
candidate_normalized = normalize(candidate_physical, physical_bounds)

posterior = model.posterior(candidate_normalized)
print("Posterior mean (original outcome units):", posterior.mean)
print("Posterior variance (squared outcome units):", posterior.variance)

# A BO optimizer using this manually normalized model returns normalized inputs.
# Convert its proposed point back before sending it to an experiment system:
# candidate_physical = unnormalize(candidate_normalized, physical_bounds)

Posterior mean (original outcome units): tensor([[1.5642]], grad_fn=<UnsqueezeBackward0>)
Posterior variance (squared outcome units): tensor([[0.0015]], grad_fn=<UnsqueezeBackward0>)


## 6. Connect the proposal to the SDL

In a full BO cycle, `optimize_acqf` proposes a point in the coordinates used to train the model. Because this example manually normalized `train_X`, the proposal is normalized too. Before sending it to the SDL:

1. Convert the proposal back to physical units with `unnormalize`.
2. Check bounds and any domain-specific formulation rules (box bounds alone do not enforce composition constraints).
3. Map values and units into the SDL's recipe/job format and execute it.
4. Read back the measured objective (and replicate-based variance if available).

This cell uses a sample normalized proposal to demonstrate the handoff; it does not submit anything to lab hardware.

In [65]:
# Example proposal in the model's normalized coordinates (not an optimized result).
proposed_X = torch.tensor([[0.60, 0.45, 0.55]])
recipe_X = unnormalize(proposed_X, physical_bounds)

salt_M, ec_fraction, dmc_fraction = recipe_X[0].tolist()
assert torch.all(recipe_X >= physical_bounds[0])
assert torch.all(recipe_X <= physical_bounds[1])

# Illustrative adapter payload only; rename fields for the actual SDL API.
sdl_job = {
    "parameters": {
        "salt_concentration_M": salt_M,
        "EC_fraction": ec_fraction,
        "DMC_fraction": dmc_fraction,
    }
}
print("Recipe to submit to SDL:", sdl_job)
# The actual experiment call belongs in your SDL-specific integration layer.

Recipe to submit to SDL: {'parameters': {'salt_concentration_M': 1.4, 'EC_fraction': 0.45, 'DMC_fraction': 0.55}}


### Return the measurement to BoTorch

After the SDL completes the job, associate its measured outcome with the exact physical recipe/job ID. Convert/store the input in the same model coordinates used for the training set, then append the real observation and refit the model:

```python
# Illustrative only: fill these from the completed SDL run.
new_Y = torch.tensor([[measured_conductivity]])
new_X_model = normalize(recipe_X, physical_bounds)
train_X = torch.cat([train_X, new_X_model])
train_Y = torch.cat([train_Y, new_Y])

model = SingleTaskGP(train_X, train_Y)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)
```

If replicate measurements provide an estimated variance, store it in outcome-squared units and keep it aligned row-for-row with `train_Y` before supplying `train_Yvar`. If a run fails, record the failure/status for the SDL workflow rather than inventing an outcome. The BO loop is therefore **propose -> convert -> execute -> measure -> update data -> refit -> propose again**.

## Summary:

- Organize data as rows of experiments: `train_X` is `n x d`; single-outcome `train_Y` is `n x 1`.
- `q` counts candidates evaluated jointly; `d` counts input variables.
- Normalize inputs exactly once and consistently use normalized or physical coordinates.
- `SingleTaskGP` standardizes outcomes by default in this setup; predictions are returned in the original outcome scale.
- `train_Yvar` contains observation variances when known, not standard deviations or posterior variances.
- Convert a normalized proposal to physical recipe values before SDL submission, then attach the returned measurement to that same run and update the training data.

**SDL loop:** physical measurements -> model data -> GP/acquisition proposal -> physical recipe -> SDL run -> new measurement -> updated model.